In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import *

In [2]:
import torch
from torch import nn # neural network
from torch.optim import Adam # parameter 최적화 알고리즘인 Adam
from torch.utils.data import TensorDataset, DataLoader # 데이터 다루는 도구

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [4]:
# 데이터 불러오기
import requests
from io import BytesIO
import certifi

path = 'https://bit.ly/advertising_csv'
response = requests.get(path,verify=certifi.where(),timeout=15)
response.raise_for_status()
adv = pd.read_csv(BytesIO(response.content))

# x, y 분할
target = 'Sales'
x = adv.drop(target, axis= 1)
y = adv.loc[:, target]

# 학습 / 검증 데이터 분리
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size= .2, random_state=20)

print(type(x_train), type(x_val)) # 판다스 데이터프레임

# 스케일링
scaler = MinMaxScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)

print(type(x_train), type(x_val)) # numpy ndarray

<class 'pandas.DataFrame'> <class 'pandas.DataFrame'>
<class 'numpy.ndarray'> <class 'numpy.ndarray'>


In [12]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Check source data
print("SOURCE DATA")
print("x_train:", type(x_train), x_train.shape)
print("y_train:", type(y_train), y_train.shape)
print("x_val:", type(x_val), x_val.shape)
print("y_val:", type(y_val), y_val.shape)

# Recreate tensors
x_train_ts = torch.as_tensor(x_train, dtype=torch.float32)
y_train_ts = torch.as_tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)

x_val_ts = torch.as_tensor(x_val, dtype=torch.float32)
y_val_ts = torch.as_tensor(y_val.to_numpy(), dtype=torch.float32).reshape(-1, 1)

# Check tensor dimensions
print("\nTENSORS")
print("x_train_ts:", x_train_ts.shape, x_train_ts.size(0))
print("y_train_ts:", y_train_ts.shape, y_train_ts.size(0))
print("x_val_ts:", x_val_ts.shape, x_val_ts.size(0))
print("y_val_ts:", y_val_ts.shape, y_val_ts.size(0))

# Verify alignment
assert x_train_ts.size(0) == y_train_ts.size(0)
assert x_val_ts.size(0) == y_val_ts.size(0)

# Create datasets
train_ds = TensorDataset(x_train_ts, y_train_ts)
val_ds = TensorDataset(x_val_ts, y_val_ts)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

print("\nSUCCESS")
print("Training samples:", len(train_ds))
print("Validation samples:", len(val_ds))

SOURCE DATA
x_train: <class 'numpy.ndarray'> (160, 3)
y_train: <class 'pandas.Series'> (160,)
x_val: <class 'numpy.ndarray'> (40, 3)
y_val: <class 'pandas.Series'> (40,)

TENSORS
x_train_ts: torch.Size([160, 3]) 160
y_train_ts: torch.Size([160, 1]) 160
x_val_ts: torch.Size([40, 3]) 40
y_val_ts: torch.Size([40, 1]) 40

SUCCESS
Training samples: 160
Validation samples: 40


C:\Users\mega\AppData\Local\Temp\ipykernel_15748\429771204.py:13: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:214.)
  y_train_ts = torch.as_tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)


In [ ]:
# 모델 선언
in_features = x_train_ts.shape[1]
model = nn.Sequential(
    nn.Linear(in_features, 1),
).to(device)

In [ ]:
# 손실함수와 옵티마이저 정의
loss_fn = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=0.01)  # 오차 최소화를 위한 모델의 가중치 조정